<a href="https://colab.research.google.com/github/The-cheater/Deep_Learning_Models/blob/main/final_correct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import zipfile
import os

# Unzip GAF Images
with zipfile.ZipFile('/content/drive/MyDrive/dataset/GAF_Images.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/GAF_Images')

# Unzip MTF Images
with zipfile.ZipFile('/content/drive/MyDrive/dataset/MTF_Images.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/MTF_Images')

# Check extracted folders
print("GAF Images:", os.listdir('/content/GAF_Images')[:5])
print("MTF Images:", os.listdir('/content/MTF_Images')[:5])


GAF Images: ['GAF_Images_train', 'GAF_Images_test']
MTF Images: ['MTF_Images_train', 'MTF_Images_test']


In [4]:
!pip install --upgrade pip
!pip install tensorflow opencv-python matplotlib
# ==========================
# 1️⃣ Imports & Config
# ==========================
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

# Label mapping
label_map = {'EL': 0, 'PD': 1, 'S': 2}



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [5]:
# ==========================
# 2️⃣ Data Loader Functions
# ==========================
def parse_pair(gaf_path, mtf_path, label):
    # Load GAF
    gaf_img = tf.io.read_file(gaf_path)
    gaf_img = tf.image.decode_png(gaf_img, channels=3)
    gaf_img = tf.image.resize(gaf_img, [224, 224])
    gaf_img = tf.cast(gaf_img, tf.float32) / 255.0

    # Load MTF
    mtf_img = tf.io.read_file(mtf_path)
    mtf_img = tf.image.decode_png(mtf_img, channels=3)
    mtf_img = tf.image.resize(mtf_img, [224, 224])
    mtf_img = tf.cast(mtf_img, tf.float32) / 255.0

    return (gaf_img, mtf_img), label

def create_file_paths_and_labels(gaf_root, mtf_root):
    gaf_paths, mtf_paths, labels = [], [], []

    for root, _, files in os.walk(gaf_root):
        for file in files:
            if file.lower().endswith('_gaf.png'):
                gaf_path = os.path.join(root, file)
                mtf_path = gaf_path.replace('GAF_Images', 'MTF_Images').replace('_gaf.png', '_mtf.png')
                if not os.path.exists(mtf_path):
                    continue  # skip if no corresponding MTF file

                # Determine label
                parts = gaf_path.split(os.sep)
                for cn in label_map.keys():
                    if cn in parts:
                        label = label_map[cn]
                        break
                else:
                    continue

                gaf_paths.append(gaf_path)
                mtf_paths.append(mtf_path)
                labels.append(label)

    return gaf_paths, mtf_paths, labels

# ==========================
# 3️⃣ Dataset Pipeline
# ==========================
from sklearn.model_selection import train_test_split

gaf_root = '/content/GAF_Images/GAF_Images_train'
mtf_root = '/content/MTF_Images/MTF_Images_train'

gaf_paths, mtf_paths, labels = create_file_paths_and_labels(gaf_root, mtf_root)
print(f"✅ Found {len(gaf_paths)} samples.")

# Split data into training and validation sets
gaf_train, gaf_val, mtf_train, mtf_val, labels_train, labels_val = train_test_split(
    gaf_paths, mtf_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"✅ Training samples: {len(gaf_train)}, Validation samples: {len(gaf_val)}")

batch_size = 16

# Create training dataset
train_dataset = tf.data.Dataset.from_tensor_slices((gaf_train, mtf_train, labels_train))
train_dataset = train_dataset.shuffle(buffer_size=10000)
train_dataset = train_dataset.map(lambda gaf, mtf, lbl: parse_pair(gaf, mtf, lbl), num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Create validation dataset
val_dataset = tf.data.Dataset.from_tensor_slices((gaf_val, mtf_val, labels_val))
val_dataset = val_dataset.map(lambda gaf, mtf, lbl: parse_pair(gaf, mtf, lbl), num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

✅ Found 8034 samples.
✅ Training samples: 6427, Validation samples: 1607


In [6]:
from tensorflow.keras import layers, Model, Input

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    return x

def channel_attention_module(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    shared_dense_one = layers.Dense(channel // ratio, activation='relu', kernel_initializer='he_normal')
    shared_dense_two = layers.Dense(channel, kernel_initializer='he_normal')
    avg_out = shared_dense_two(shared_dense_one(avg_pool))
    max_out = shared_dense_two(shared_dense_one(max_pool))
    cbam_feature = layers.Add()([avg_out, max_out])
    cbam_feature = layers.Activation('sigmoid')(cbam_feature)
    cbam_feature = layers.Reshape((1, 1, channel))(cbam_feature)
    return layers.Multiply()([input_feature, cbam_feature])

# Inputs
input_gaf = Input(shape=(224, 224, 3), name='gaf_input')
input_mtf = Input(shape=(224, 224, 3), name='mtf_input')

# GAF stream (as per diagram)
x1 = layers.Conv2D(64, 3, padding='same', activation='relu')(input_gaf)      # 224x224x64
x1 = conv_block(x1, 128)                                                     # 112x112x128
x1 = conv_block(x1, 256)                                                     # 56x56x256
x1 = conv_block(x1, 256)                                                     # 28x28x256

# MTF stream (as per diagram)
x2 = layers.Conv2D(64, 3, padding='same', activation='relu')(input_mtf)      # 224x224x64
x2 = conv_block(x2, 128)                                                     # 112x112x128
x2 = conv_block(x2, 256)                                                     # 56x56x256
x2 = conv_block(x2, 256)                                                     # 28x28x256

# Merge features (concatenate)
merged = layers.Concatenate(axis=-1)([x1, x2])                               # 28x28x512

# Further convolution and pooling to reach 14x14x512
x = layers.Conv2D(512, 3, padding='same', activation='relu')(merged)         # 28x28x512
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)                                 # 14x14x512
x = layers.Conv2D(512, 3, padding='same', activation='relu')(x)              # 14x14x512
x = layers.BatchNormalization()(x)

# Apply Channel Attention Module at 14x14x512
x = channel_attention_module(x)

# Flatten and classify (no 7x7x4096, no further pooling)
x = layers.Flatten()(x)
output = layers.Dense(3, activation='softmax')(x)

# Build and compile model
model = Model(inputs=[input_gaf, input_mtf], outputs=output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ gaf_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mtf_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │      1,792 │ gaf_input[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 224, 224,  │      1,792 │ mtf_input[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │     73,856 │ conv2d[0][0]      │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 224, 224,  │     73,856 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 224, 224,  │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        512 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │    295,168 │ max_pooling2d[0]… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 112, 112,  │    295,168 │ max_pooling2d_3[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │      1,024 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │      1,024 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 56, 56,    │    590,080 │ max_pooling2d_1[

 Total params: 7,017,795 (26.77 MB)

 Trainable params: 7,013,187 (26.75 MB)

 Non-trainable params: 4,608 (18.00 KB)

In [7]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_accuracy',    # Monitor validation accuracy
    patience=5,                # Stop after 5 epochs with no improvement
    restore_best_weights=True  # Restore weights from the epoch with the best val_accuracy
)
history = model.fit(
    train_dataset,
    epochs=30,
    validation_data=val_dataset,
    callbacks=[early_stop]
)


Epoch 1/30
402/402 ━━━━━━━━━━━━━━━━━━━━ 252s 525ms/step - accuracy: 0.4588 - loss: 4.6946 - val_accuracy: 0.4256 - val_loss: 1.4204
Epoch 2/30
402/402 ━━━━━━━━━━━━━━━━━━━━ 208s 459ms/step - accuracy: 0.5724 - loss: 1.1763 - val_accuracy: 0.5825 - val_loss: 1.1898
Epoch 3/30
402/402 ━━━━━━━━━━━━━━━━━━━━ 195s 441ms/step - accuracy: 0.6663 - loss: 0.8469 - val_accuracy: 0.6777 - val_loss: 0.7565
Epoch 4/30
402/402 ━━━━━━━━━━━━━━━━━━━━ 209s 459ms/step - accuracy: 0.7862 - loss: 0.5224 - val_accuracy: 0.7803 - val_loss: 0.5451
Epoch 5/30
402/402 ━━━━━━━━━━━━━━━━━━━━ 202s 459ms/step - accuracy: 0.8631 - loss: 0.3492 - val_accuracy: 0.8363 - val_loss: 0.4076
Epoch 6/30
 47/402 ━━━━━━━━━━━━━━━━━━━━ 2:24 406ms/step - accuracy: 0.9190 - loss: 0.2293

KeyboardInterrupt: 

In [ ]:
model.save('/content/drive/MyDrive/my_model.h5')


In [ ]:
from tensorflow import keras

# Load the saved model
model = keras.models.load_model('/content/drive/MyDrive/my_model.h5')


In [ ]:
# 📌 Upload Images from Local Machine
from google.colab import files
from IPython.display import display
from PIL import Image
import tensorflow as tf
import numpy as np

print("✅ Please upload the GAF image (RGB)")
uploaded_gaf = files.upload()
gaf_path = list(uploaded_gaf.keys())[0]

print("✅ Please upload the MTF image (RGB)")
uploaded_mtf = files.upload()
mtf_path = list(uploaded_mtf.keys())[0]

# Display the images
display(Image.open(gaf_path))
display(Image.open(mtf_path))

# 🖼️ Preprocessing
def preprocess_image(img_path):
    img = Image.open(img_path).convert('RGB')
    img = img.resize((224, 224))
    img_array = np.array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

gaf_test = preprocess_image(gaf_path)
mtf_test = preprocess_image(mtf_path)

print("✅ Preprocessed shapes:", gaf_test.shape, mtf_test.shape)

# ✅ Load the trained model
model = tf.keras.models.load_model('/content/mmcnn_model_final.h5')
print("✅ Model loaded.")

# 🔮 Predict
predictions = model.predict([gaf_test, mtf_test])
predicted_class = np.argmax(predictions, axis=1)[0]
confidence = np.max(predictions)

# Mapping back to class name
class_map = {0: 'EL', 1: 'PD', 2: 'S'}
print(f"✅ Predicted Class: {class_map[predicted_class]} with Confidence: {confidence:.4f}")
